In [1]:
# =========================
# 0) Install (Colab)
# =========================
# !pip install -U torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
# !pip install -U unsloth
# !pip install -U transformers==4.56.2 datasets==4.3.0
# !pip install -U --no-deps trl==0.22.2

import torch
assert torch.cuda.is_available(), "Enable GPU runtime (Colab → Runtime → GPU)"

In [2]:
# =========================
# 1) OOP: Unsloth FineTuner
# =========================
from dataclasses import dataclass
from typing import Optional, List
from datasets import load_dataset
from unsloth import FastLanguageModel
from peft import PeftModel
from trl import SFTTrainer, SFTConfig


c:\Users\vibhu\anaconda3\envs\cain\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0723 23:03:25.239000 7688 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
@dataclass
class FineTuneConfig:
    # model
    model_name: str
    load_in_4bit: bool = True
    max_seq_length: int = 4096
    dtype: Optional[str] = None  # None = auto

    # dataset
    dataset_name: str = "tatsu-lab/alpaca"
    split: str = "train"
    seed: int = 3407

    # training
    output_dir: str = "outputs"
    lora_save_path: str = "lora_adapters"
    per_device_bs: int = 32
    grad_acc_steps: int = 32
    epochs: int = 1
    lr: float = 2e-5
    warmup_ratio: float = 0.1
    logging_steps: int = 10
    packing: bool = True

    # lora
    lora_r: int = 32
    lora_alpha: int = 32
    lora_dropout: float = 0.0
    target_modules: List[str] = None
    use_gc: bool = False

    # save merged model
    save_merged: bool = False
    merged_save_path: str = "merged_fp16_model"

In [4]:
class UnslothFineTuner:
    def __init__(self, cfg: FineTuneConfig):
        self.cfg = cfg
        if self.cfg.target_modules is None:
            self.cfg.target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
        self.model = None
        self.tokenizer = None

    # ---------- Load Model ----------
    def load_model(self):
        print(f"✅ Loading model: {self.cfg.model_name}")

        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name=self.cfg.model_name,
            max_seq_length=self.cfg.max_seq_length,
            dtype=self.cfg.dtype,
            load_in_4bit=self.cfg.load_in_4bit,
        )

        self.model = FastLanguageModel.get_peft_model(
            self.model,
            r=self.cfg.lora_r,
            target_modules=self.cfg.target_modules,
            lora_alpha=self.cfg.lora_alpha,
            lora_dropout=self.cfg.lora_dropout,
            bias="none",
            use_gradient_checkpointing=self.cfg.use_gc,
            random_state=self.cfg.seed,
        )

        self.model.print_trainable_parameters()
        print("Is PEFT model?", isinstance(self.model, PeftModel))
        print("Device:", next(self.model.parameters()).device, " | Dtype:", next(self.model.parameters()).dtype)

    # ---------- Load Dataset ----------
    def load_dataset(self):
        print(f"✅ Loading dataset: {self.cfg.dataset_name} (split={self.cfg.split})")

        # Try HF load first, then local
        try:
            ds = load_dataset(self.cfg.dataset_name, split=self.cfg.split)
        except Exception:
            ds = load_dataset(path=self.cfg.dataset_name, split=self.cfg.split)

        ds = ds.shuffle(seed=self.cfg.seed)
        print(ds)
        print("Columns:", ds.column_names)
        return ds

    # ---------- Formatting Helpers ----------
    def _eos(self):
        return self.tokenizer.eos_token or "</s>"

    def _alpaca_prompt(self):
        return """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

    def _format_alpaca(self, batch):
        EOS = self._eos()
        prompt = self._alpaca_prompt()
        texts = []
        inp_list = batch.get("input", [""] * len(batch["instruction"]))
        for ins, inp, out in zip(batch["instruction"], inp_list, batch["output"]):
            texts.append(prompt.format(
                instruction=ins or "",
                input=inp or "",
                output=out or ""
            ) + EOS)
        return {"text": texts}

    def _format_dolly(self, batch):
        EOS = self._eos()
        prompt = self._alpaca_prompt()
        ctx_list = batch.get("context", [""] * len(batch["instruction"]))
        texts = []
        for ins, ctx, resp in zip(batch["instruction"], ctx_list, batch["response"]):
            texts.append(prompt.format(
                instruction=ins or "",
                input=ctx or "",
                output=resp or ""
            ) + EOS)
        return {"text": texts}

    def _format_sharegpt(self, batch):
        EOS = self._eos()
        texts = []

        conv_key = None
        if "conversations" in batch:
            conv_key = "conversations"
        elif "messages" in batch:
            conv_key = "messages"
        else:
            raise ValueError("ShareGPT format needs 'conversations' or 'messages' column.")

        for conv in batch[conv_key]:
            if conv is None:
                texts.append("" + EOS)
                continue

            turns = []
            for t in conv:
                if isinstance(t, dict) and "from" in t and "value" in t:
                    role, content = t["from"], t["value"]
                elif isinstance(t, dict) and "role" in t and "content" in t:
                    role, content = t["role"], t["content"]
                else:
                    continue
                turns.append((role, content))

            chat = ""
            for role, content in turns:
                if role in ["human", "user"]:
                    chat += f"### User:\n{content}\n\n"
                else:
                    chat += f"### Assistant:\n{content}\n\n"

            texts.append(chat.strip() + EOS)

        return {"text": texts}

    def _format_text(self, batch):
        EOS = self._eos()
        if "text" not in batch:
            raise ValueError("Expected a 'text' column.")
        return {"text": [(t or "") + EOS for t in batch["text"]]}

    def _format_pharma_custom(self, batch):
        """
        For sunny199/pharma-instruction-data OR pharma_instruction_data
        Supports common patterns:
        - instruction,input,output
        - question,answer
        - prompt,completion
        - input,output
        """
        EOS = self._eos()
        cols = set(batch.keys())

        if {"instruction", "output"}.issubset(cols):
            return self._format_alpaca(batch)

        if {"question", "answer"}.issubset(cols):
            texts = []
            for q, a in zip(batch["question"], batch["answer"]):
                texts.append(f"### Question:\n{q or ''}\n\n### Answer:\n{a or ''}" + EOS)
            return {"text": texts}

        if {"prompt", "completion"}.issubset(cols):
            texts = []
            for p, c in zip(batch["prompt"], batch["completion"]):
                texts.append(f"{p or ''}\n{c or ''}" + EOS)
            return {"text": texts}

        if {"input", "output"}.issubset(cols):
            prompt = self._alpaca_prompt()
            texts = []
            for i, o in zip(batch["input"], batch["output"]):
                texts.append(prompt.format(
                    instruction="Answer the following:",
                    input=i or "",
                    output=o or ""
                ) + EOS)
            return {"text": texts}

        raise ValueError(f"Pharma formatter can't infer columns: {sorted(list(cols))}")

    # ---------- Infer dataset format ----------
    def format_dataset(self, ds):
        print("✅ Inferring dataset format...")

        name = self.cfg.dataset_name
        cols = set(ds.column_names)

        # known dataset mappings
        if name == "tatsu-lab/alpaca":
            print("✅ Format: ALPACA")
            return ds.map(self._format_alpaca, batched=True, remove_columns=ds.column_names)

        if name == "databricks/databricks-dolly-15k":
            print("✅ Format: DOLLY")
            return ds.map(self._format_dolly, batched=True, remove_columns=ds.column_names)

        if name == "anon8231489123/ShareGPT_Vicuna_unfiltered":
            print("✅ Format: SHAREGPT")
            return ds.map(self._format_sharegpt, batched=True, remove_columns=ds.column_names)

        if name == "OpenAssistant/oasst1":
            print("✅ Format: OASST (auto)")
            if "text" in cols:
                return ds.map(self._format_text, batched=True, remove_columns=ds.column_names)
            return ds.map(self._format_sharegpt, batched=True, remove_columns=ds.column_names)

        if name in ["sunny199/pharma-instruction-data", "pharma_instruction_data"]:
            print("✅ Format: PHARMA (custom)")
            return ds.map(self._format_pharma_custom, batched=True, remove_columns=ds.column_names)

        # fallback heuristics
        if "text" in cols:
            print("✅ Format: TEXT (fallback)")
            return ds.map(self._format_text, batched=True, remove_columns=ds.column_names)

        if "conversations" in cols or "messages" in cols:
            print("✅ Format: CHAT (fallback)")
            return ds.map(self._format_sharegpt, batched=True, remove_columns=ds.column_names)

        if {"instruction", "output"}.issubset(cols):
            print("✅ Format: ALPACA-LIKE (fallback)")
            return ds.map(self._format_alpaca, batched=True, remove_columns=ds.column_names)

        raise ValueError(f"❌ Could not infer dataset format. Columns: {sorted(list(cols))}")

    # ---------- Train ----------
    def train(self, ds):
        print("✅ Starting training...")
        trainer = SFTTrainer(
            model=self.model,
            tokenizer=self.tokenizer,
            train_dataset=ds,
            dataset_text_field="text",
            packing=self.cfg.packing,
            args=SFTConfig(
                per_device_train_batch_size=self.cfg.per_device_bs,
                gradient_accumulation_steps=self.cfg.grad_acc_steps,
                num_train_epochs=self.cfg.epochs,
                learning_rate=self.cfg.lr,
                warmup_ratio=self.cfg.warmup_ratio,
                optim="adamw_8bit",
                logging_steps=self.cfg.logging_steps,
                seed=self.cfg.seed,
                output_dir=self.cfg.output_dir,
                report_to="none",
            ),
        )
        trainer.train()

    # ---------- Save ----------
    def save(self):
        print("✅ Saving LoRA adapters to:", self.cfg.lora_save_path)
        self.model.save_pretrained(self.cfg.lora_save_path)
        self.tokenizer.save_pretrained(self.cfg.lora_save_path)

        if self.cfg.save_merged:
            print("✅ Merging LoRA + saving full model to:", self.cfg.merged_save_path)
            merged = self.model.merge_and_unload()
            merged.save_pretrained(self.cfg.merged_save_path, safe_serialization=True)
            self.tokenizer.save_pretrained(self.cfg.merged_save_path)

    # ---------- Full pipeline ----------
    def run(self):
        self.load_model()
        raw_ds = self.load_dataset()
        ds = self.format_dataset(raw_ds)
        print("✅ Sample formatted text:\n", ds["text"][0][:800])
        self.train(ds)
        self.save()
        print("✅ Done!")




In [5]:
# =========================
# 2) USE IT (Object)
# =========================
cfg = FineTuneConfig(
    model_name="unsloth/gemma-3-1b-it-bnb-4bit",
    dataset_name="OpenAssistant/oasst1",
    split="train",
    lora_save_path="gemma-3-1b-it_lora",
    output_dir="gemma-3-1b-it_outputs",
    epochs=1,
    per_device_bs=1,
    grad_acc_steps=64,
    lr=2e-5,
    max_seq_length=1024,
    use_gc="unsloth",
    save_merged=False,  # True if you want merged fp16 model
)

trainer = UnslothFineTuner(cfg)
trainer.run()

✅ Loading model: unsloth/gemma-3-1b-it-bnb-4bit
==((====))==  Unsloth 2026.7.5: Fast Gemma3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce GTX 1650. Num GPUs = 1. Max memory: 4.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.7.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
trainable params: 26,091,520 || all params: 1,025,977,472 || trainable%: 2.5431
Is PEFT model? True
Device: cuda:0  | Dtype: torch.float16
✅ Loading dataset: OpenAssistant/oasst1 (split=train)
Dataset({
    features: ['message_id', 'parent_id', 'user_id', 'created_date

Map: 100%|██████████| 84437/84437 [00:01<00:00, 50782.30 examples/s]


✅ Sample formatted text:
 En general, para la observación planetaria, se recomienda el uso de un telescopio reflector debido a su capacidad para capturar imágenes más nítidas y detalladas de los planetas. Los telescopios reflectores son capaces de recolectar más luz que los refractores del mismo tamaño, lo que les permite proporcionar una imagen más brillante y detallada.

Además, los telescopios reflectores suelen tener una apertura más grande en relación con su longitud focal que los telescopios refractores de tamaño comparable, lo que les permite obtener imágenes más brillantes y con mayor resolución.

Sin embargo, en áreas con alta contaminación lumínica, la elección del telescopio puede no ser tan crucial como la elección del lugar de observación. En estos casos, se recomienda buscar un lugar lo más alejado po
✅ Starting training...
Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"]: 100%|██████████| 84437/84437 [00:22<00:00, 3755.09 examples/s]


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 84,437 | Num Epochs = 1 | Total steps = 1,320
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 64
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 64 x 1) = 64
 "-____-"     Trainable parameters = 26,091,520 of 1,025,977,472 (2.54% trained)


Step,Training Loss
10,2.679300
20,2.613300
30,2.516100
40,2.503400
50,2.509300
60,2.364100
70,2.327300
80,2.332900
90,2.140600
100,2.267600


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!
✅ Saving LoRA adapters to: gemma-3-1b-it_lora
✅ Done!


In [6]:
import torch
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory/1024**3)
import unsloth, transformers
print(unsloth.__version__)
print(transformers.__version__)

NVIDIA GeForce GTX 1650
3.99969482421875
2026.7.5
4.56.2


In [1]:
from unsloth import FastLanguageModel

# Load your adapter and base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="gemma-3-1b-it_lora",  # Path to your saved LoRA adapters
    max_seq_length=1024,
    load_in_4bit=False,  # Unquantize for clean evaluation export
)

# Save merged 16-bit model for OpenCompass evaluation
model.save_pretrained_merged(
    "gemma-3-1b-it_merged", tokenizer, save_method="merged_16bit"
)
print("✅ Merged model successfully saved to ./gemma-3-1b-it_merged")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\Users\vibhu\anaconda3\envs\cain\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0804 22:30:16.375000 22284 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.5: Fast Gemma3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce GTX 1650. Num GPUs = 1. Max memory: 4.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.7.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 6f78ee0f-a270-41ec-9e49-83870caf86b0)')' thrown while requesting HEAD https://huggingface.co/unsloth/gemma-3-1b-it/resolve/main/adapter_config.json
[huggingface_hub.utils._http|WARNING]'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 6f78ee0f-a270-41ec-9e49-83870caf86b0)')' thrown while requesting HEAD https://huggingface.co/unsloth/gemma-3-1b-it/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].
[huggingface_hub.utils._http|WARNING]Retrying in 1s [Retry 1/5].


Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.
Found HuggingFace hub cache directory: C:\Users\vibhu\.cache\huggingface\hub
Checking cache directory for required files...


...

Successfully copied all 1 files from cache to `gemma-3-1b-it_merged`
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `gemma-3-1b-it_merged`: 100%|██████████| 1/1 [00:00<00:00, 333.23it/s]


Successfully copied all 1 files from cache to `gemma-3-1b-it_merged`


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:08<00:00,  8.07s/it]


Unsloth: Merge process complete. Saved to `d:\SLM finetune\gemma-3-1b-it_merged`
✅ Merged model successfully saved to ./gemma-3-1b-it_merged


In [2]:
import opencompass
print(opencompass.__version__)
print(opencompass.__path__)

0.5.3
['c:\\Users\\vibhu\\anaconda3\\envs\\cain\\Lib\\site-packages\\opencompass']


In [ ]:
# cfg = FineTuneConfig(
#     model_name="unsloth/gemma-3-1b-it-bnb-4bit",
#     dataset_name="tatsu-lab/alpaca",
# )

## 3) Test the Model
Now we load the saved LoRA adapters and run inference to verify the fine-tuning results.

In [1]:
from unsloth import FastLanguageModel
import torch

# 1. Load the model and tokenizer (must match the base model used in training)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "gemma-3-1b-it_merged", # Path where we saved the adapters
    max_seq_length = 1024,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model) # Enable 2x faster inference

def generate_response(instruction, input_text=""):
    prompt = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
""".format(instruction, input_text)

    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)[0].split("### Response:")[-1].strip()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\Users\vibhu\anaconda3\envs\cain\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.4.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(
c:\Users\vibhu\anaconda3\envs\cain\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0806 11:45:35.662000 13812 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.5: Fast Gemma3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce GTX 1650. Num GPUs = 1. Max memory: 4.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.7.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


In [6]:
messages = [
    {
        "role": "user",
        "content": "A farmer has 17 sheep. All but 9 die. How many remain?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(
    inputs,
    max_new_tokens=256,
    do_sample=False,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

user
A farmer has 17 sheep. All but 9 die. How many remain?
model
Let the total number of sheep be 17.
The number of sheep that survive is 17 - 9 = 8.


In [5]:
# Example Test Prompt
test_instruction = "There are 20 sheep, all but 3 die, how many are left?"
print(f"Prompt: {test_instruction}")
print("-" * 30)
print(generate_response(test_instruction))

Prompt: There are 20 sheep, all but 3 die, how many are left?
------------------------------
If you have the case of 4, and shying feelings are similar to 3, putting All to All could be another option.
The other method is a new loop that starts at 7. 
Each iteration makes each cell a 4 5 8 iteration.

!!! Tell me if this has the same answer, if I get further corrupt AI, then I will delete the entire iteration.
        *(Because I know it should be 6. What do you tell me.)*

!!! Explain and then answer if you give me a different answer!
   *(Because I know it should be 4. What do you tell me?)*
   *(Because the only method that begins with 5, 8, and 4 works.)*/




                                  
                                         !!! Explain give me more of the explanation and then answer the 5. 
                                      !!! Tell me more of your explanation AND then answer me.)*/
   *(Don't ignore me! Come back to me, I can learn and think in your style.)*/
   *(D

## 4) Model Specifications & Details
Let's inspect the LoRA adapter configuration and the model's base architecture details.

In [ ]:
import json
import os

# Path to the saved LoRA adapters
lora_path = "phi3_pharma_lora"
config_path = os.path.join(lora_path, "adapter_config.json")

print("--- Model Architecture & LoRA Config ---")
if os.path.exists(config_path):
    with open(config_path, "r") as f:
        config_data = json.load(f)

    print(f"Base Model: {config_data.get('base_model_name_or_path')}")
    print(f"Peft Type: {config_data.get('peft_type')}")
    print(f"LoRA Rank (r): {config_data.get('r')}")
    print(f"LoRA Alpha: {config_data.get('lora_alpha')}")
    print(f"Target Modules: {config_data.get('target_modules')}")
    print(f"LoRA Dropout: {config_data.get('lora_dropout')}")
else:
    print("Adapter config not found.")

print("\n--- Runtime Details ---")
print(f"Model Data Type: {model.dtype}")
print(f"Model Device: {model.device}")
print(f"Max Sequence Length: {model.config.max_position_embeddings}")
print(f"Vocabulary Size: {len(tokenizer)}")

In [ ]:
import os

def get_dir_size(path='.'):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)
    return total_size

# Calculate parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# Disk size
lora_size_mb = get_dir_size('phi3_pharma_lora') / (1024 * 1024)

print(f"--- Model Size Details ---")
print(f"Total Parameters: {total_params:,}")
print(f"Trainable (LoRA) Parameters: {trainable_params:,}")
print(f"Base Model Class: Phi-3-mini (3.8B class)")
print(f"LoRA Adapter Disk Size: {lora_size_mb:.2f} MB")